# A/B test: AI model v1 vs v2

**Decision:** Should we ship model v2 to all patients?

**Primary metric:** resolution rate of AI conversations.
**Guardrails:** escalation rate on symptom concerns, urgent clinical escalations,
QA-reviewed answer accuracy on clinical questions.

**Decision rule (set before looking):** Ship only if resolution improves AND no
safety guardrail gets significantly worse.

In [1]:
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import chisquare

raw = "../data/raw/"
exposures = pd.read_csv(raw + "experiment_exposures.csv", parse_dates=["first_exposure_at"])
convos = pd.read_csv(raw + "conversations.csv", parse_dates=["started_at"])
qa = pd.read_csv(raw + "qa_reviews.csv")

In [2]:
convos.head(5)

,conversation_id,patient_id,started_at,model_version,intent,confidence,escalated,escalation_type,resolved,csat
0,1,5324,2024-04-01 01:37:00,v1,shipping,0.867,True,support_agent,True,NaN
1,2,9855,2024-04-01 07:57:00,v1,side_effects,0.645,True,clinician,True,NaN
2,3,2571,2024-04-01 08:25:00,v1,side_effects,0.685,False,NaN,True,5.0
3,4,17510,2024-04-01 08:35:00,v1,shipping,0.986,False,NaN,True,NaN
4,5,424,2024-04-01 09:06:00,v1,shipping,0.735,True,support_agent,True,4.0


In [3]:
counts = exposures["variation_name"].value_counts()
stat, p = chisquare(counts, f_exp=[counts.sum() / 2] * 2)
print(counts.to_dict(), f"SRM p = {p:.3f}")

{'v1': 2361, 'v2': 2350} SRM p = 0.873


In [4]:
print("rows before dedup:", len(convos))
convos = convos.drop_duplicates()
print("rows after dedup:", len(convos))

rows before dedup: 64163
rows after dedup: 63844


In [5]:
start, end = pd.Timestamp("2025-08-01"), pd.Timestamp("2025-11-01")

In [6]:
in_window = convos[(convos["started_at"] >= start) & (convos["started_at"] < end)]
exp = in_window.merge(exposures[["patient_id", "variation_name"]], on="patient_id")

In [7]:
exp["v2"] = (exp["variation_name"] == "v2").astype(int)
exp["resolved"] = exp["resolved"].astype(int)
exp["escalated"] = exp["escalated"].astype(int)
exp["urgent"] = (exp["escalation_type"] == "urgent_clinical").astype(int)

In [8]:
print(len(exp), "conversations from", exp["patient_id"].nunique(), "patients")
pd.crosstab(exp["variation_name"], exp["model_version"])

13522 conversations from 4711 patients


model_version,v1,v2
variation_name,,
v1,6804,0
v2,0,6718


In [11]:
def compare(df,metric):
    """v1 vs v2 on a 0/1 metric. Standard errors are clustered by patient,
    because we randomized patients but measure conversations."""
    model = smf.ols(f"{metric} ~ v2", data = df).fit(cov_type = "cluster", 
                                                     cov_kwds = {"groups":df["patient_id"]})
    v1 = df.loc[df["v2"] == 0, metric].mean()
    lo, hi = model.conf_int().loc["v2"]
    return pd.Series({"v1": v1, "v2": v1 + model.params["v2"],
                      "diff": model.params["v2"], "ci_low": lo, "ci_high": hi,
                      "p": model.pvalues["v2"], "n_convos": len(df)})

In [12]:
compare(exp, "resolved")

v1              0.707084
v2              0.728342
diff            0.021258
ci_low          0.005621
ci_high         0.036894
p               0.007710
n_convos    13522.000000
dtype: float64

In [13]:
symptom = exp[exp["intent"] == "symptom_concern"]

results = pd.DataFrame({
    "Primary: resolution rate": compare(exp, "resolved"),
    "Escalation rate (all)": compare(exp, "escalated"),
    "Guardrail: escalation on symptom concerns": compare(symptom, "escalated"),
    "Guardrail: urgent clinical escalations": compare(exp, "urgent"),
}).T
results.round(4)

,v1,v2,diff,ci_low,ci_high,p,n_convos
Primary: resolution rate,0.7071,0.7283,0.0213,0.0056,0.0369,0.0077,13522.0
Escalation rate (all),0.2864,0.2551,-0.0313,-0.0473,-0.0153,0.0001,13522.0
Guardrail: escalation on symptom concerns,0.7250,0.6823,-0.0426,-0.0881,0.0028,0.0657,1598.0
Guardrail: urgent clinical escalations,0.0403,0.0409,0.0007,-0.0066,0.0079,0.8578,13522.0


In [14]:
clinical = ["dosing_question", "side_effects", "symptom_concern"]
qa_exp = exp.merge(qa[["conversation_id", "true_intent", "answer_correct"]],
                   on="conversation_id")
qa_exp["clinical"] = qa_exp["true_intent"].isin(clinical)
qa_exp["answer_correct"] = qa_exp["answer_correct"].astype(int)

qa_exp.groupby(["clinical", "variation_name"])["answer_correct"].agg(["mean", "count"])

mean  count
clinical variation_name                 
False    v1              0.771930    171
         v2              0.905759    191
True     v1              0.542254    142
         v2              0.579710    138

1. Does v2 win on the primary metric? Look at the difference and whether the CI excludes zero.
2. Does v2 escalate less overall? Is that good (efficiency) or bad (safety)? It depends on which conversations stop getting escalated.
3. Look at the symptom-concern and urgent rows closely. Fewer escalations there means fewer patients with possible symptoms reaching a clinician.
4. Check the QA counts. The QA sample is only 5%, so clinical accuracy may rest on a small number of reviews with wide error bars. That's a finding in itself: the safety monitoring may be too thin to catch problems, so increase QA sampling for clinical conversations in any v2 rollout.

### Ans: v2 wins on primary metric of resolution rate since this increases ~2pts, the CI excludes zero on this metric and the pvalue ~ 0.008. The escalation rate does fall as well, but looking at the guardrail metrics, the esclation decreases on symptoms concerns by ~4 pts which is more than the overall rate, and stays relatively the same for urgent clinical escalations. For the guardrail metrics both CIs overlap with 0 which is concerning. The first guardrail of escaltion of symptoms, is probably underpowered and the sample size isn't large enough.

### What to do instead: When designing this, a better guardrail design is non-inferiority test or not letting the guardrail metric decrease by this much, so a symptom concern escalation can't drop by more than 2 points.

### Reccomendation: Wouldn't ship v2 with these results.
### Recommendation: don't ship v2 to everyone. Instead:

Ship v2 for logistics intents (refill, shipping, billing), where it's better.
Keep v1 or force escalation for clinical intents until safety is shown.
Increase QA sampling on clinical conversations so the safety metric has enough data.

In [17]:
plain = smf.ols("resolved ~ v2", data=exp).fit()
print("unclustered CI:", plain.conf_int().loc["v2"].round(4).tolist())
print("clustered CI:  ", compare(exp, "resolved")[["ci_low", "ci_high"]].round(4).tolist())

unclustered CI: [0.0061, 0.0364]
clustered CI:   [0.0056, 0.0369]


In [18]:
calib = qa_exp.groupby(["clinical", "variation_name"]).agg(
    mean_confidence=("confidence", "mean"),
    accuracy=("answer_correct", "mean"),
    n=("answer_correct", "size"))
calib["overconfidence"] = calib["mean_confidence"] - calib["accuracy"]
calib.round(3)

mean_confidence  accuracy    n  overconfidence
clinical variation_name                                                
False    v1                        0.874     0.772  171           0.102
         v2                        0.918     0.906  191           0.012
True     v1                        0.687     0.542  142           0.145
         v2                        0.735     0.580  138           0.155

#### v2 improves resolution by 2.1 points, driven by much better logistics answers.
#### It escalates symptom concerns about 4 points less often (CI includes up to a 9-point drop).
#### QA can't confirm clinical safety because the sample is far too small, and [calibration result] suggests the drop comes from overconfidence.
#### Recommendation: ship v2 for logistics intents. Keep clinical intents on v1 or force escalation. Oversample clinical conversations for QA (stratified, not 5% random) until you can run a powered non-inferiority tes

## Recommendation

**Result:** v2 improves resolution by 2.1 pts (95% CI 0.6–3.7), driven by
better and well-calibrated logistics answers (accuracy 77% → 91%;
overconfidence gap 10 pts → 1 pt).

**Risk:** v2 escalates symptom concerns 4.3 pts less (CI −8.8 to +0.3).
Its clinical confidence rose ~5 pts without a measurable accuracy gain,
and escalation is confidence-driven. Both models are ~15 pts overconfident
on clinical questions.

**Limits:** QA covers ~140 clinical reviews per arm; detecting a 3-pt
accuracy difference would need ~4,300 per arm. The symptom-concern
guardrail was underpowered.

**Recommendation:**
1. Ship v2 for logistics intents (refill, shipping, billing).
2. Keep clinical intents on v1 behavior, or force escalation, for now.
3. Recalibrate confidence for clinical intents (e.g., isotonic calibration
   or per-intent escalation thresholds) — this affects v1 today too.
4. Replace 5% random QA with stratified sampling that oversamples clinical
   conversations, and gate clinical v2 on a powered non-inferiority test.